In [8]:
import os
import re
import pandas as pd

# ----------------------------
# 1. Define the full grid
# ----------------------------
envs = {
    "CP": "CartPole-v1",
    "AC": "Acrobot-v1",
    "LL": "LunarLander-v2",
}

seeds = [42, 43, 123, 456, 789, 999, 1280, 1290, 2024, 2025, 2344, 2345, 6788, 6789]
topologies = ["STANDARD_MLP", "SW", "MOD", "HYB", "FC"]
sizes = ["S128", "S256", "S384"]
nodes = ["N001", "N002", "N003"]

# Build full grid
grid = pd.DataFrame([
    (envs[env_key], seed, topo, size, node)
    for env_key in envs
    for seed in seeds
    for topo in topologies
    for size in sizes
    for node in nodes
], columns=["env", "seed", "topology", "size", "node"])

# ----------------------------
# 2. Parse completed runs
# ----------------------------
pattern = re.compile(
    r"(?P<topology>[A-Z_]+)_.*_S(?P<size>\d+)_.*_(?P<env>CP|AC|LL)_seed(?P<seed>\d+)_.*_N(?P<node>\d{4})"
)

completed = []

root = "test_experiments"  # <-- change if needed
for dirpath, dirnames, filenames in os.walk(root):
    for dirname in dirnames:
        m = pattern.match(dirname)
        if m:
            topo = m.group("topology")
            size = f"S{m.group('size')}"
            env = envs[m.group("env")]
            seed = int(m.group("seed"))
            node = f"N{m.group('node')[-3:]}"  # e.g. N0002 → N002
            completed.append((env, seed, topo, size, node))

completed_df = pd.DataFrame(completed, columns=["env", "seed", "topology", "size", "node"])

# ----------------------------
# 3. Compare grid vs completed
# ----------------------------
merged = grid.merge(completed_df.assign(done=True),
                    on=["env", "seed", "topology", "size", "node"],
                    how="left")
merged["status"] = merged["done"].apply(lambda x: "✅" if x else "❌")

# ----------------------------
# 4. Save outputs: one CSV per topology
# ----------------------------
for topo in topologies:
    df_topo = merged[merged["topology"] == topo]

    pivot = df_topo.pivot_table(
        index=["env", "seed"],
        columns=["node", "size"],
        values="status",
        aggfunc="first"
    )

    filename = f"completed_runs_{topo}.csv"
    pivot.to_csv(filename)
    print(f"Saved {filename}")

# CSV with missing runs only
missing = merged[merged["status"] == "❌"]
missing.to_csv("missing_runs.csv", index=False)

print("Done. Outputs saved: completed_runs_<TOPO>.csv for each topology, and missing_runs.csv")


Saved completed_runs_STANDARD_MLP.csv
Saved completed_runs_SW.csv
Saved completed_runs_MOD.csv
Saved completed_runs_HYB.csv
Saved completed_runs_FC.csv
Done. Outputs saved: completed_runs_<TOPO>.csv for each topology, and missing_runs.csv


In [9]:
merged_pivot

topology                   FC            HYB            MOD            \
size                     S128 S256 S384 S128 S256 S384 S128 S256 S384   
env            seed node                                                
Acrobot-v1     42   N001    ✅    ✅    ✅    ✅    ✅    ✅    ✅    ✅    ✅   
                    N002    ✅    ✅    ✅    ✅    ✅    ✅    ✅    ✅    ✅   
                    N003    ✅    ✅    ✅    ✅    ✅    ✅    ✅    ✅    ✅   
               43   N001    ✅    ✅    ✅    ✅    ✅    ✅    ✅    ✅    ✅   
                    N002    ✅    ✅    ✅    ✅    ✅    ✅    ✅    ✅    ✅   
...                       ...  ...  ...  ...  ...  ...  ...  ...  ...   
LunarLander-v2 6788 N002    ✅    ✅    ✅    ✅    ✅    ✅    ✅    ✅    ✅   
                    N003    ✅    ✅    ✅    ✅    ✅    ✅    ✅    ✅    ✅   
               6789 N001    ✅    ✅    ✅    ✅    ✅    ✅    ✅    ✅    ✅   
                    N002    ✅    ✅    ✅    ✅    ✅    ✅    ✅    ✅    ✅   
                    N003    ✅    ✅    ✅    ✅    ✅    ✅    ✅    ✅    ✅   

topology                 STANDARD_MLP             SW            
size                             S128 S256 S384 S128 S256 S384  
env            seed node                                        
Acrobot-v1     42   N001            ✅    ✅    ✅    ✅    ✅    ✅  
                    N002            ✅    ✅    ✅    ✅    ✅    ✅  
                    N003            ✅    ✅    ✅    ✅    ✅    ✅  
               43   N001            ✅    ✅    ✅    ✅    ✅    ✅  
                    N002            ✅    ✅    ✅    ✅    ✅    ✅  
...                               ...  ...  ...  ...  ...  ...  
LunarLander-v2 6788 N002            ✅    ✅    ✅    ✅    ✅    ✅  
                    N003            ✅    ✅    ✅    ✅    ✅    ✅  
               6789 N001            ✅    ✅    ✅    ✅    ✅    ✅  
                    N002            ✅    ✅    ✅    ✅    ✅    ✅  
                    N003            ✅    ✅    ✅    ✅    ✅    ✅  

[126 rows x 15 columns]

In [10]:
missing

,env,seed,topology,size,node,done,status
